# DP-OT: Public→Private Graph Transfer — Synthetic experiments

Train a GNN on a **public** source graph, then adapt to a **private** target graph using only differentially-private prototype-mass estimates (edge-DP). The final model is DP by post-processing.

**This notebook = synthetic only.** Real-data experiments (OGB-arxiv) are in `colab_real_data.ipynb`.

**How to use:** Runtime → *Run all*. Cells go: clone → install → single experiment → diagnostic → corrected-DGP experiment → sweep → plots.

Recommended: **Runtime → Change runtime type → GPU** (T4 is plenty). The notebook auto-detects and uses it.

## 1. Clone the repo
Edit `BRANCH` if you merged this work into a different branch. Re-running pulls the latest commit instead of failing.

In [ ]:
import os

REPO_URL = "https://github.com/ChefAltoids/MM-edgeDP"
BRANCH   = "dp-ot"          # the branch this code lives on
REPO_DIR = "/content/MM-edgeDP"

if not os.path.isdir(REPO_DIR):
    !git clone --branch $BRANCH $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git fetch origin && git checkout $BRANCH && git pull --ff-only

os.chdir(REPO_DIR)          # so that dp_ot/outputs/... relative paths work
print("cwd:", os.getcwd())

## 2. Install dependencies
Colab ships PyTorch + CUDA already. We only add PyTorch Geometric (and `ogb`, used by the optional real-data loader).

In [ ]:
import torch
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())

# Core PyG is enough for SAGEConv on dense graphs (no need for torch-scatter wheels).
!pip install -q torch_geometric
# Only needed if you run the OGB-arxiv fallback; harmless otherwise.
!pip install -q ogb
print("\nInstall complete.")

## 3. Imports + device
`DEVICE` is threaded into every config below, so the whole pipeline runs on GPU when one is attached.

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from dp_ot.run_experiment import run_experiment
from dp_ot.sweep import run_sweep
from dp_ot.eval import plots

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

def show_results(results: dict) -> pd.DataFrame:
    """Tidy a run_experiment() dict into a sorted DataFrame."""
    df = pd.DataFrame(results).T[["auroc", "acc", "f1", "proto_l1_error"]]
    order = ["source_only", "dp_histogram", "dp_exponential", "oracle", "target_oracle"]
    return df.reindex([m for m in order if m in df.index]).round(4)

## 4. Single experiment (synthetic covariate shift)
One seed, one (γ, ε, K). Expected ordering of AUROC:
`source_only` < `dp_histogram` ≈ `dp_exponential` < `oracle` < `target_oracle`.

In [ ]:
cfg = {
    "dataset": "synthetic",
    "n_source": 1000, "n_target": 500, "M": 4, "d_latent": 8,
    "gamma": 0.5,        # covariate-shift magnitude
    "epsilon": 1.0,      # edge-DP budget
    "K": 32,             # number of prototypes
    "seed": 0,
    "epochs": 200,
    "device": DEVICE,
}

results = run_experiment(cfg)
show_results(results)

### 4b. Adaptation diagnostic — is this config in an interesting regime?
Prints the non-private **oracle gain** over source-only and the fraction each DP method recovers. A near-zero gap means there is nothing to recover (overlapping support + flexible model) — switch to the `support_mismatch` regime or raise the shift.

In [ ]:
def adaptation_diagnostic(results):
    so  = results['source_only']['auroc']
    orc = results['oracle']['auroc']
    tgt = results['target_oracle']['auroc']
    gap = orc - so
    print(f'source_only   AUROC : {so:.4f}')
    print(f'oracle        AUROC : {orc:.4f}  (gain over source: {gap:+.4f})')
    print(f'target_oracle AUROC : {tgt:.4f}  (upper bound)')
    if gap <= 1e-3:
        print('  -> oracle does NOT beat source-only: nothing for DP to recover.')
        print('     Try regime=support_mismatch, higher gamma, or lower model capacity.')
        return
    for m in ['dp_histogram', 'dp_exponential']:
        a  = results[m]['auroc']
        l1 = results[m]['proto_l1_error']
        rec = (a - so) / gap
        print(f'  {m:15s}: recovers {rec*100:6.1f}% of oracle gain  (AUROC {a:.4f}, proto_l1 {l1:.3f})')

adaptation_diagnostic(results)

### 4c. Corrected DGP — balanced-label, misspecified covariate shift
The earlier regimes produced *degenerate* targets (one Gaussian → saturated, ~constant labels → AUROC at chance), so `oracle ≈ source` told us nothing. This regime fixes the data-generating process so adaptation can actually be tested:

- **balanced labels:** component means are orthogonalized against the label axis, so every cluster is ~50/50 (watch the printed class balance — majority should be near 50%, not 90%+);
- **misspecification:** a per-cluster sign flip makes `P(Y|Z)` XOR-like, so the source-optimal predictor differs from the target-optimal one;
- **support shift + limited capacity:** source under-covers the target's clusters and the model is small, so reweighting toward them measurably raises target AUROC.

Expect a **clearly positive oracle gain** here. If it's still ~0, the knobs to turn are in the discussion below the cell.

In [ ]:
cfg_ms = {
    'dataset': 'synthetic', 'regime': 'misspec_support',
    'n_source': 1500, 'n_target': 600, 'M': 6, 'd_latent': 8,
    'mean_scale': 2.5,             # separated clusters (not so far it saturates)
    'gamma': 0.85,                 # target concentrates on the favored clusters
    'source_minority_mass': 0.08,  # source barely covers them
    'label_spread': 3.0,           # separable BUT balanced label signal
    'label_sharpness': 1.5,
    'K': 32, 'B': 4.0, 'epsilon': 1.0, 'seed': 0,
    'hidden': 8, 'epochs': 60, 'weight_decay': 0.01,  # capacity-limited
    'device': DEVICE,
}
results_ms = run_experiment(cfg_ms)   # prints source/target class balance
display(show_results(results_ms))
adaptation_diagnostic(results_ms)

## 5. Sweep (ε × γ × seed)
`run_sweep` expands the `sweep` block, writes a resumable CSV, and returns a DataFrame. Start small; widen the grid once it looks right. On a GPU each combination is a few seconds.

> The grid below is `5 ε × 3 γ × 3 seeds = 45` combinations. Bump the lists for publication-quality error bars. To sweep the support-mismatch regime instead, add `'regime': 'support_mismatch'`, `'mean_scale': 4.0`, `'source_minority_mass': 0.05`, `'hidden': 8`, `'epochs': 80` and set `gamma: [0.9]`.

In [ ]:
sweep_cfg = {
    "dataset": "synthetic",
    "n_source": 1000, "n_target": 500, "M": 4, "d_latent": 8,
    "epochs": 200,
    "device": DEVICE,
    "out": "dp_ot/outputs/sweep_colab.csv",
    "sweep": {
        "epsilon": [0.1, 0.3, 1.0, 3.0, 10.0],
        "gamma":   [0.0, 0.5, 1.0],
        "K":       [32],
        "seed":    [0, 1, 2],
    },
}

df = run_sweep(sweep_cfg)
df.tail(10)

## 6. Plots
Three figures: AUROC vs ε (at γ=0.5), AUROC vs γ (at ε=1.0), and prototype-mass L1 recovery vs ε. They are also saved as PDFs under `dp_ot/outputs/`.

In [ ]:
df = pd.read_csv("dp_ot/outputs/sweep_colab.csv")
df.columns = df.columns.str.strip()

plots.plot_auroc_vs_epsilon(df, gamma=0.5, out_path="dp_ot/outputs/auroc_vs_epsilon.pdf")
plots.plot_auroc_vs_gamma(df, epsilon=1.0, out_path="dp_ot/outputs/auroc_vs_gamma.pdf")
plots.plot_l1_vs_epsilon(df, gamma=0.5, out_path="dp_ot/outputs/l1_vs_epsilon.pdf")
plt.show()

## 7. Save outputs
Colab disks are ephemeral. Download the CSV + figures, or mount Drive to persist them.

*(Real-data experiments now live in their own notebook, `colab_real_data.ipynb`, so synthetic and real results stay separate.)*

In [ ]:
import shutil
shutil.make_archive("/content/dp_ot_outputs", "zip", "dp_ot/outputs")

try:
    from google.colab import files
    files.download("/content/dp_ot_outputs.zip")
except Exception as e:
    print("Download skipped (not in Colab?):", e)

# --- Alternatively, persist to Google Drive ---
# from google.colab import drive
# drive.mount('/content/drive')
# shutil.copytree('dp_ot/outputs', '/content/drive/MyDrive/dp_ot_outputs', dirs_exist_ok=True)